# RMSNorm

root mean square normalization (RMSNorm) 是一种归一化技术，他是相较于于 LayerNorm 的另一种选择，在 Transformer 模型中，RMSNorm 被广泛应用。layerNorm相较于baseline能够在相同训练steps下加速模型收敛，即在相同steps的情况下，layernorm能够帮助loss下降更加快速。但是layernorm的计算开销较大，相同训练时间下loss下降并没有那么明显。layernorm能够稳定训练的一个重要特性是其重新中心化不变形（re-centering invariance），简单来说就是当数据发生整体偏移时，由于均值和方差的归一化方法，layernorm之后的结果可以将整体偏移消除。<br>
假设输入向量为：$x=[x_1, x_2, x_3, ... , x_d]$ <br>
layernorm 公式为：
$$
\mu = \frac{1}{d} \sum_{i=1}^{d} x_i; 
$$
$$
\sigma = \sqrt{\frac{1}{d} \sum_{i=1}^{d} (x_i - \mu)^2}; 
$$
$$
y_i = \frac{x_i - \mu}{\sigma}
$$


而 RMSNorm 认为均值归一化并不会减少隐藏状态或模型梯度的方差，因此推测这一操作对 LayerNorm 的成功并不起关键作用。<br>
即$x \rightarrow x - \mu$ 即减去均值（re-centering）不会降低hidden state的方差，因为:<br>
$$
Var(x - \mu) = Var(x)
$$
所以，RMSNorm 的公式为：
$$
\sigma = \sqrt{\frac{1}{d} \sum_{i=1}^{d} x_i^2}; 
$$
$$
y_i = \frac{x_i}{\sigma} *g
$$
<br>
RMSNorm 的优点是计算开销小，并且能够加速模型收敛。

In [1]:
import torch
import torch.nn as nn


In [ ]:
class RMSNorm(nn.Module):
    def __init__(self, hidden_size, eps=1e-5):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(hidden_size))
        self.variance_epsilon = eps

    def forward(self, hidden_states: torch.Tensor):
        input_dtype = hidden_states.dtype
        hidden_states = hidden_states.to(torch.float32)
        variance = hidden_states.pow(2).mean(-1, keepdim=True)
        hidden_states = hidden_states * torch.rsqrt(variance + self.variance_epsilon)
        hidden_states = self.weight * hidden_states.to(input_dtype)
        return hidden_states
